# 3 — From Graph to Spreadsheet and Back

In the previous notebook we built the Swiss garden system by writing Python code: each node and edge was an explicit Python object. In the real world, LCA data almost always arrives as a **spreadsheet** (Excel, CSV, or similar). This notebook teaches you how to:

1. Understand the **bill-of-materials (BOM)** paradigm and why it differs from a graph
2. Export a Brightway database to Excel and inspect it
3. Solve the **matching problem** — linking spreadsheet rows back to database nodes
4. Use the `bw2io` **ETL pipeline** (Extract → Transform → Load) to import the data back
5. Handle unlinked edges gracefully

## Using LLMs in this notebook

Use Claude, ChatGPT, or Copilot freely for the exercises. Good prompts include:

- *"Using bw2io ExcelImporter, how do I apply a custom strategy function?"*
- *"What does link_iterable_by_fields do when fields=['name', 'location']?"*
- *"How do I iterate over sheets in an openpyxl workbook and find a cell by value?"*

Paste error messages directly — LLMs diagnose them quickly.

---

## Section 1 — The Bill-of-Materials Paradigm

### Graphs use pointers; spreadsheets use attributes

When you build a graph in Python, an edge from process A to process B contains a **direct pointer** to the B object. There is no ambiguity:

```python
a.new_edge(input=b, amount=1.0, type="technosphere")
```

A spreadsheet (bill of materials) cannot contain Python pointers. Instead, each row describes an exchange by listing the **attributes** of the thing it references:

| name | location | unit | amount |
|---|---|---|---|
| gravel | CH | kg | 3200 |
| natural stone plate | CH | kg | 1000 |

To turn those rows back into a graph, we must find the database node whose name, location, and unit **match** those attributes. This is called **linking** (or matching).

### Graph vs. BOM — side by side

| What | Graph | BOM (spreadsheet) |
|---|---|---|
| Edge from A to B | `A.new_edge(input=B, amount=1)` | Row: name="B", location="CH", amount=1 |
| Identifying B | Python object reference | Match name + location + unit against a database |
| Risk of error | None — pointer is always correct | High if names, units, or locations don't match exactly |

### Why matching is hard

There is **no guarantee** that the attributes in a spreadsheet uniquely identify a node:

- Two databases may spell the same substance differently (`"CO2"` vs. `"Carbon dioxide"`)
- Units might differ (`"kg"` vs. `"kilogram"`)
- Location codes might differ (`"RER"` vs. `"Europe"`)
- The same name may refer to multiple processes in different contexts

Brightway's `bw2io` library handles this through an **ETL pipeline**: Extract, Transform, Load.

---

## Section 2 — Exporting the Swiss Garden to Excel

In [1]:
import bw2data as bd
import bw2io as bi
from pathlib import Path

# Use the project created in the Swiss garden notebook
bd.projects.set_current("BAFU 2025 v2")

fg = bd.Database("Swiss garden foreground")
print(f"Foreground database has {len(fg)} nodes")
list(bd.databases)

Foreground database has 10 nodes


['BAFU-2025-biosphere', 'BAFU-2025-v2', 'Swiss garden foreground']

`bw2io` can write any Brightway database to an Excel file in a format that can be re-imported:

In [2]:
# Export to Excel — returns the path of the written file
filepath = bi.export.write_lci_excel(
    'Swiss garden foreground',
    dirpath=str(Path.cwd()),
)
print(f'Written to {filepath}')

Written to /Users/cmutel/Code/brightway/eta-teaching/lci-Swiss-garden-foreground.xlsx


### Exercise 1

Open the exported file with excel or `pandas` and inspect it. Answer these questions:

1. What columns are present in the first sheet?
2. What does each **row** represent — a node or an exchange?
3. How are the foreground processes separated from the exchanges inside them?
4. Can you see the background inputs (e.g., gravel, excavation)? What information identifies them?

In [3]:
import pandas as pd

# Inspect the exported file — it uses a vertical block format:
# each process is a block starting with 'Activity', followed by metadata rows,
# then exchange rows
df = pd.read_excel(filepath, sheet_name=0, header=None)
df.head(40)

,0,1,2,3,4,5,6
0,Database,Swiss garden foreground,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Activity,Annual garden maintenance,NaN,NaN,NaN,NaN,NaN
3,code,annual_maintenance_proc,NaN,NaN,NaN,NaN,NaN
4,location,CH,NaN,NaN,NaN,NaN,NaN
5,reference product,Garden maintenance,NaN,NaN,NaN,NaN,NaN
6,type,process,NaN,NaN,NaN,NaN,NaN
7,unit,unit,NaN,NaN,NaN,NaN,NaN
8,worksheet name,Swiss garden foreground,NaN,NaN,NaN,NaN,NaN
9,Exchanges,NaN,NaN,NaN,NaN,NaN,NaN


---

## Section 3 — Deliberately Breaking the Data

Real-world imports almost always have imperfections. To make this exercise realistic, we will introduce **two deliberate problems** that commonly arise:

1. **Name inconsistency**: rename `"Garden construction"` → `"Garden construction process"` (a typical naming drift between data sources)
2. **Unit mismatch**: change the unit of one exchange from `"unit"` → `"pieces"` (a common discrepancy when combining data from different systems)

We will save the modified file as `swiss_garden_broken.xlsx` so the original is preserved.

In [4]:
# Introduce two deliberate imperfections into a copy of the file
import openpyxl, shutil

# Work on a copy so we keep the clean original
broken_path = Path.cwd() / 'swiss_garden_broken.xlsx'
shutil.copy(filepath, broken_path)

wb = openpyxl.load_workbook(broken_path)
ws = wb.active

# Problem 1: rename 'Garden construction' → 'Garden construction process'
for row in ws.iter_rows():
    for cell in row:
        if cell.value == 'Garden construction':
            cell.value = 'Garden construction process'
            print(f'  Renamed at row {cell.row}')

# Problem 2: change the first 'unit' in an exchange row to 'pieces'
# Exchange rows have 'technosphere' or 'production' in col 5 (index 4)
changed = False
for row in ws.iter_rows():
    cells_vals = [c.value for c in row]
    if not changed and cells_vals[3] == 'unit' and cells_vals[4] in ('technosphere', 'production'):
        row[3].value = 'pieces'
        changed = True
        print(f'  Unit changed at row {row[0].row}')

wb.save(broken_path)
print(f'Saved broken version to {broken_path}')

  Unit changed at row 104
Saved broken version to /Users/cmutel/Code/brightway/eta-teaching/swiss_garden_broken.xlsx


---

## Section 4 — The ETL Pipeline

ETL stands for **Extract → Transform → Load**. Every import in `bw2io` follows this pattern:

```
Excel file  ──► ExcelImporter  ──► apply_strategies()  ──► match_database()  ──► write_database()
              (Extract)           (Transform)              (Load prep)           (Load)
```

### Step 1 — Extract

In [5]:
imp = bi.ExcelImporter(broken_path)
# Look at the first extracted dataset (raw, before any transformation)
imp.data[0]

Extracted 1 worksheets in 0.01 seconds


{'code': 'annual_maintenance_proc',
 'location': 'CH',
 'reference product': 'Garden maintenance',
 'type': 'process',
 'unit': 'unit',
 'worksheet name': 'Swiss garden foreground',
 'name': 'Annual garden maintenance',
 'exchanges': [{'name': 'Garden maintenance',
   'amount': 1,
   'location': 'CH',
   'unit': 'unit',
   'type': 'production'},
  {'name': 'Average mineral fertiliser, as K2O, at regional storehouse',
   'amount': 0.25,
   'location': 'RER',
   'unit': 'kilogram',
   'categories': 'agricultural::agricultural means of production\\mineral fertiliser',
   'type': 'technosphere',
   'comment': 'K₂O fertiliser (kg)'},
  {'name': 'Average mineral fertiliser, as N, at regional storehouse',
   'amount': 0.75,
   'location': 'RER',
   'unit': 'kilogram',
   'categories': 'agricultural::agricultural means of production\\mineral fertiliser',
   'type': 'technosphere',
   'comment': 'N fertiliser for 50 m² lawn (kg N)'},
  {'name': 'Average mineral fertiliser, as P2O5, at regional 

Notice that the extracted data is already structured as a list of dictionaries — one per process. Each dictionary contains an `'exchanges'` key with a list of exchange dictionaries. No linking has happened yet: the exchanges have `name`, `location`, `unit` attributes but **no `input` pointer**.

### Exercise 2

How many datasets (processes) were extracted? How many total exchanges? Use `imp.statistics()` to find out, and check the number manually by inspecting `imp.data`.

In [6]:
# Your code here
imp.statistics()

Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
0 edges to the following databases:
31 unique unlinked edges (33 total):
	technosphere: 26
	production: 5




(10, 33, 33, 0)

### Step 2 — Transform (default strategies)

The `ExcelImporter` comes with a list of default transformation strategies. These handle common tasks like normalising units, ensuring categories are stored as tuples, and attempting internal linking.

In [7]:
# See the default strategy list before applying anything
imp.strategies

[<function bw2io.strategies.csv.csv_restore_tuples(data)>,
 <function bw2io.strategies.csv.csv_restore_booleans(data)>,
 <function bw2io.strategies.csv.csv_numerize(data)>,
 <function bw2io.strategies.csv.csv_drop_unknown(data)>,
 <function bw2io.strategies.csv.csv_restore_temporal_distributions(data)>,
 <function bw2io.strategies.csv.csv_add_missing_exchanges_section(data)>,
 <function bw2io.strategies.generic.normalize_units(db: List[dict]) -> List[dict]>,
 <function bw2io.strategies.biosphere.strip_biosphere_exc_locations(db)>,
 <function bw2io.strategies.generic.set_code_by_activity_hash(db: List[dict], overwrite: bool = False) -> List[dict]>,
 functools.partial(<function link_iterable_by_fields at 0x145aa4510>, other=Brightway2 SQLiteBackend: biosphere3, edge_kinds=['biosphere']),
 <function bw2io.strategies.generic.assign_only_product_as_production(db: Iterable[dict]) -> List[dict]>,
 <function bw2io.strategies.generic.link_technosphere_by_activity_hash(db, external_db_name: str 

In [8]:
imp.apply_strategies()
imp.statistics()

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_restore_temporal_distributions
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 15 strategies in 0.00 seconds
Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
8 edges to the following databases:
	Swiss garden foreground: 8
23 unique unlinked edges (25 total):
	techn

(10, 33, 25, 0)

### Exercise 3

After applying the default strategies, some exchanges are still **unlinked**. 

1. What does "unlinked" mean? (Think about what field is missing from an unlinked exchange dictionary.)
2. Are the unlinked exchanges from the foreground or the background?
3. Look at them using the cell below — can you guess why they are not linked?

In [9]:
# Inspect the unlinked exchanges
# imp.unlinked is a generator — wrap in list() to see all
for edge in imp.unlinked:
    print(edge)

{'name': 'Average mineral fertiliser, as K2O, at regional storehouse', 'amount': 0.25, 'location': 'RER', 'unit': 'kilogram', 'categories': ('agricultural', 'agricultural means of production\\mineral fertiliser'), 'type': 'technosphere', 'comment': 'K₂O fertiliser (kg)'}
{'name': 'Average mineral fertiliser, as N, at regional storehouse', 'amount': 0.75, 'location': 'RER', 'unit': 'kilogram', 'categories': ('agricultural', 'agricultural means of production\\mineral fertiliser'), 'type': 'technosphere', 'comment': 'N fertiliser for 50 m² lawn (kg N)'}
{'name': 'Average mineral fertiliser, as P2O5, at regional storehouse', 'amount': 0.15, 'location': 'RER', 'unit': 'kilogram', 'categories': ('agricultural', 'agricultural means of production\\mineral fertiliser'), 'type': 'technosphere', 'comment': 'P₂O₅ fertiliser (kg)'}
{'name': 'Electricity, low voltage, certified electricity, at grid', 'amount': 10, 'location': 'CH', 'unit': 'kilowatt hour', 'categories': ('electricity', 'supply mix')

### Step 3 — Fix the name problem

One of the unlinked exchanges references `"Garden construction process"` — the name we deliberately broke. The default strategies cannot fix this because they don't know what the correct name should be. We need to write a **custom transformation function**.

A transformation function:
- Takes `data` (a list of dataset dicts) as its only argument
- Modifies the data in place (or returns a modified copy)
- Returns the list

In [10]:
def fix_construction_name(data):
    """Rename 'Garden construction process' back to 'Garden construction'."""
    for ds in data:
        for exc in ds['exchanges']:
            if exc.get('name') == 'Garden construction process':
                exc['name'] = 'Garden construction'
    return data

imp.apply_strategy(fix_construction_name)
imp.statistics()

Applying strategy: fix_construction_name
Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
8 edges to the following databases:
	Swiss garden foreground: 8
23 unique unlinked edges (25 total):
	technosphere: 22
	production: 1




(10, 33, 25, 0)

### Exercise 4

The unit mismatch is still causing an unlinked exchange. Write your own transformation function to fix it:
- Find any exchange whose `unit` is `"pieces"`
- Change it back to `"unit"`

Apply it with `imp.apply_strategy(your_function)` and verify with `imp.statistics()` that the unlinked count decreases.

In [11]:
# Write your fix_unit_mismatch function here
# Hint: loop over ds['exchanges'], check if exc['unit'] == 'pieces', fix it

def fix_unit_mismatch(data):
    # Your code here
    return data

imp.apply_strategy(fix_unit_mismatch)
imp.statistics()

Applying strategy: fix_unit_mismatch
Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
8 edges to the following databases:
	Swiss garden foreground: 8
23 unique unlinked edges (25 total):
	technosphere: 22
	production: 1




(10, 33, 25, 0)

### Step 4 — Linking

With names and units fixed, we can now link the foreground exchanges against the foreground database. `match_database` is a convenience wrapper around `link_iterable_by_fields` (explored in depth in the next section).

In [12]:
# Link internal foreground references: name + location should be sufficient
# because we control the foreground and know names are unique there
imp.match_database("Swiss garden foreground", fields=['name', 'location'])
imp.statistics()

Applying strategy: link_iterable_by_fields
Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
9 edges to the following databases:
	Swiss garden foreground: 9
22 unique unlinked edges (24 total):
	technosphere: 22




(10, 33, 24, 0)

### Exercise 5

Why do we match against `"Swiss garden foreground"` first, and not `"BAFU-2025-v2"`? 

Now try also linking against the background database. What happens to the unlinked count? Do you need additional fields for BAFU?

In [13]:
# Try linking against the background database
imp.match_database("BAFU-2025-v2", fields=['name', 'location'])
imp.statistics()

Applying strategy: link_iterable_by_fields


Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
33 edges to the following databases:
	BAFU-2025-v2: 24
	Swiss garden foreground: 9
0 unique unlinked edges (0 total):




(10, 33, 0, 0)

### Exercise 6

Some background inputs reference BAFU processes by name and location. Would `name + location` always be sufficient to uniquely identify a BAFU node? Think about:

- Can two BAFU processes have the same name and location but different reference products?
- What additional field(s) could you add to make the match more specific?
- What happens if you use **too many** fields (e.g., you include a field that doesn't exist in one database)?

Experiment with different `fields` combinations.

In [14]:
# Experiment here: try different field combinations
# e.g. fields=['name', 'location', 'unit']
# or   fields=['name', 'location', 'reference product']

# Check which edges are still unlinked
for edge in imp.unlinked:
    print(edge)

---

## Section 5 — `link_iterable_by_fields` in Depth

`match_database` is a convenience wrapper. The underlying workhorse is `link_iterable_by_fields`. Understanding it lets you handle cases that `match_database` can't.

### What it does

For each exchange in `unlinked` that lacks an `input` field:

1. Build a **key** from the exchange's values at the requested `fields`
2. Search `other` for nodes with the same key
3. If **exactly one** match is found, set `exchange['input'] = (database, code)` of that node
4. If **zero or more than one** match is found, leave the exchange unlinked (no error is raised)

The function is **conservative**: it never guesses.

In [15]:
from bw2io.strategies import link_iterable_by_fields
from copy import deepcopy

# A minimal example: one process, three exchanges
unlinked_data = [
    {
        "name": "Steel Production",
        "database": "my_db",
        "code": "steel_001",
        "exchanges": [
            {
                "name": "Iron Ore",
                "location": "Australia",
                "amount": 1.5,
                "type": "technosphere",
                "unit": "kg"
            },
            {
                "name": "Coal",
                "location": "China",
                "amount": 0.8,
                "type": "technosphere",
                "unit": "kg"
            },
            {
                "name": "Carbon Dioxide",
                "categories": ("air",),
                "amount": 2.3,
                "type": "biosphere",
                "unit": "kg"
            }
        ]
    }
]

target_database = [
    {"name": "Iron Ore", "location": "Australia", "database": "ecoinvent", "code": "iron_ore_au"},
    {"name": "Iron Ore", "location": "Brazil",    "database": "ecoinvent", "code": "iron_ore_br"},
    {"name": "Carbon Dioxide", "categories": ("air",), "database": "biosphere3", "code": "co2_air"}
]

# Link on name + location — this will match Iron Ore (Australia) but NOT Iron Ore (Brazil)
# because Brazil is not in our exchange list, and NOT Carbon Dioxide (no location field)
result = link_iterable_by_fields(
    deepcopy(unlinked_data),
    other=target_database,
    fields=["name", "location"]
)
result

[{'name': 'Steel Production',
  'database': 'my_db',
  'code': 'steel_001',
  'exchanges': [{'name': 'Iron Ore',
    'location': 'Australia',
    'amount': 1.5,
    'type': 'technosphere',
    'unit': 'kg',
    'input': ('ecoinvent', 'iron_ore_au')},
   {'name': 'Coal',
    'location': 'China',
    'amount': 0.8,
    'type': 'technosphere',
    'unit': 'kg'},
   {'name': 'Carbon Dioxide',
    'categories': ('air',),
    'amount': 2.3,
    'type': 'biosphere',
    'unit': 'kg',
    'input': ('biosphere3', 'co2_air')}]}]

Notice that Carbon Dioxide is still unlinked — it has no `location` field, so the key built from `["name", "location"]` does not match anything in the target. We need a separate call with the right fields for biosphere flows:

In [16]:
# First pass: link technosphere exchanges by name + location
result = link_iterable_by_fields(
    deepcopy(unlinked_data),
    other=target_database,
    fields=["name", "location"],
    edge_kinds=["technosphere"]   # only process technosphere exchanges
)

# Second pass: link biosphere exchanges by name + categories
result = link_iterable_by_fields(
    result,
    other=target_database,
    fields=["name", "categories"],
    edge_kinds=["biosphere"]       # only process biosphere exchanges
)

result

[{'name': 'Steel Production',
  'database': 'my_db',
  'code': 'steel_001',
  'exchanges': [{'name': 'Iron Ore',
    'location': 'Australia',
    'amount': 1.5,
    'type': 'technosphere',
    'unit': 'kg',
    'input': ('ecoinvent', 'iron_ore_au')},
   {'name': 'Coal',
    'location': 'China',
    'amount': 0.8,
    'type': 'technosphere',
    'unit': 'kg'},
   {'name': 'Carbon Dioxide',
    'categories': ('air',),
    'amount': 2.3,
    'type': 'biosphere',
    'unit': 'kg',
    'input': ('biosphere3', 'co2_air')}]}]

### The `internal=True` parameter

When all datasets to link are in the **same list** (foreground-to-foreground links), pass `internal=True` instead of `other=`:

In [17]:
internal_data = [
    {
        "name": "System A", "database": "fg", "code": "A",
        "exchanges": [
            {"name": "System B", "location": "CH", "amount": 1, "type": "technosphere"}
        ]
    },
    {
        "name": "System B", "location": "CH", "database": "fg", "code": "B",
        "exchanges": []
    }
]

link_iterable_by_fields(
    internal_data,
    internal=True,       # other=internal_data is implied
    fields=["name", "location"]
)

[{'name': 'System A',
  'database': 'fg',
  'code': 'A',
  'exchanges': [{'name': 'System B',
    'location': 'CH',
    'amount': 1,
    'type': 'technosphere',
    'input': ('fg', 'B')}]},
 {'name': 'System B',
  'location': 'CH',
  'database': 'fg',
  'code': 'B',
  'exchanges': []}]

### `edge_kinds`, `this_node_kinds`, `other_node_kinds`

These three parameters let you **filter** which exchanges and which nodes participate in the linking:

| Parameter | Filters | Typical values |
|---|---|---|
| `edge_kinds` | Which **exchange types** to process | `["technosphere"]`, `["biosphere"]` |
| `this_node_kinds` | Which **source node types** to process | `["process"]`, `["chimaera"]` |
| `other_node_kinds` | Which **target node types** may be linked to | `["product"]`, `["process"]` |

Example — link technosphere exchanges from process nodes to product nodes only (preventing process→process links):

```python
link_iterable_by_fields(
    data,
    internal=True,
    fields=["name"],
    edge_kinds=["technosphere"],
    this_node_kinds=["process"],
    other_node_kinds=["product"]
)
```

### Exercise 7 — Linking with multiple field combinations

Given the following `unlinked_data` and `target_db`, write the `link_iterable_by_fields` call(s) that link **all** technosphere and biosphere exchanges. Note that:

- Technosphere exchanges have a `location` field; use `name + location`
- Biosphere exchanges have a `categories` field; use `name + categories`

After your calls, verify that every exchange has an `input` key.

In [18]:
unlinked_ex7 = [
    {
        "name": "Electricity Production, Natural Gas",
        "location": "DE",
        "database": "student_db",
        "code": "elec_ng_de",
        "exchanges": [
            {"name": "Natural Gas", "location": "RU", "unit": "MJ",  "amount": 1.2, "type": "technosphere"},
            {"name": "Natural Gas", "location": "NO", "unit": "MJ",  "amount": 0.3, "type": "technosphere"},
            {"name": "Methane",     "categories": ("air",), "unit": "kg", "amount": 0.04, "type": "biosphere"},
            {"name": "Carbon Dioxide", "categories": ("air", "unspecified"), "unit": "kg", "amount": 0.5, "type": "biosphere"},
        ]
    }
]

target_ex7 = [
    {"name": "Natural Gas", "location": "RU", "database": "ecoinvent", "code": "ng_ru"},
    {"name": "Natural Gas", "location": "NO", "database": "ecoinvent", "code": "ng_no"},
    {"name": "Methane",     "categories": ("air",), "database": "biosphere3", "code": "ch4_air"},
    {"name": "Carbon Dioxide", "categories": ("air", "unspecified"), "database": "biosphere3", "code": "co2_unspec"},
]

In [19]:
# Your solution here — use link_iterable_by_fields with edge_kinds
# Call it once for technosphere edges, once for biosphere edges
# from bw2io.strategies import link_iterable_by_fields (already imported)

# Your code here
# result_ex7 = link_iterable_by_fields(...)

# Verify: every exchange should have an 'input' key
# for exc in result_ex7[0]['exchanges']:
#     print(exc.get('name'), '->', exc.get('input', 'UNLINKED'))

---

## Section 6 — Handling Unlinked Edges

After all transformation and linking steps, you may still have unlinked exchanges. There are three strategies:

### Option A — Create a new biosphere database

For biosphere flows that are not in any existing database. The new flows get a code but no characterisation factors, so they contribute zero to LCIA scores (unless you add CFs manually):

```python
imp.create_new_biosphere("my-new-biosphere-db")
```

### Option B — Add to an existing biosphere database

If you have an existing biosphere database and want to extend it:

```python
imp.add_unlinked_flows_to_biosphere_database("existing-biosphere-db")
```

### Option C — Drop unlinked edges

Delete the unlinked exchanges entirely. This is **destructive** — the affected flows disappear from the inventory:

```python
imp.drop_unlinked(i_am_reckless=True)
```

The `i_am_reckless=True` flag is intentional: it forces you to acknowledge that you know what you are doing.

### Exercise 8

Think through the following scenario: you import a foreground system and it has three unlinked biosphere exchanges (CO₂, CH₄, NOₓ). You choose `drop_unlinked(i_am_reckless=True)`.

1. What happens to the LCIA score for a climate-change method?
2. What happens to the LCIA score for an air-quality method?
3. Would dropping unlinked **technosphere** exchanges also affect the score? Why?
4. When might dropping unlinked edges be acceptable?

Write your reasoning in the cell below (as a comment or markdown).

In [20]:
# Your analysis here (as comments):
#
# 1. Climate score:
# 2. Air quality score:
# 3. Unlinked technosphere:
# 4. When acceptable:

---

## Section 7 — Loading (Writing the Database)

Once all exchanges are linked (or you have decided how to handle unlinked ones), write the data to Brightway.

In [21]:
# Check that everything is linked before writing
print(f"All linked: {imp.all_linked}")
imp.statistics()

Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
33 edges to the following databases:
	BAFU-2025-v2: 24
	Swiss garden foreground: 9
0 unique unlinked edges (0 total):


All linked: True
Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
33 edges to the following databases:
	BAFU-2025-v2: 24
	Swiss garden foreground: 9
0 unique unlinked edges (0 total):




(10, 33, 0, 0)

In [22]:
# If there are unlinked biosphere edges, create a new database for them
# (skip this block if all_linked is True)
if not imp.all_linked:
    imp.create_new_biosphere("Swiss garden new biosphere")

# Write — this creates (or overwrites) the database in the current project
# The importer will ask for a database name if one isn't set
# Change the database name in the Excel header or set it explicitly:
# imp.db_name = "Swiss garden foreground (imported)"
imp.write_database()

Graph statistics for `Swiss garden foreground` importer:
10 graph nodes:
	process: 5
	product: 5
33 graph edges:
	technosphere: 28
	production: 5
33 edges to the following databases:
	BAFU-2025-v2: 24
	Swiss garden foreground: 9
0 unique unlinked edges (0 total):




  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 43062.67it/s]

00:40:58+0200 [info     ] Vacuuming database            


Created database: Swiss garden foreground


In [23]:
# Verify the database exists in the project
bd.databases

Databases dictionary with 3 object(s):
	BAFU-2025-biosphere
	BAFU-2025-v2
	Swiss garden foreground

In [24]:
# Quick sanity check: count nodes in the newly imported database
# (replace the name with whatever was set above)
imported = bd.Database("Swiss garden foreground")
print(f"Imported database has {len(imported)} nodes")

Imported database has 10 nodes


### Exercise 9 — Capstone

Go back to notebook 1 (the supply chain graph) where you built a **steel bicycle** (or another small system). Export it to Excel, then import it back into Brightway following the full ETL pipeline from this notebook:

1. Export with `bi.export.write_lci_excel()`
2. Load with `bi.ExcelImporter()`
3. Apply default strategies
4. Inspect `imp.statistics()` — how many unlinked exchanges?
5. Write custom transformation functions to fix any problems
6. Match against the relevant databases
7. Write the database

Document each step and any problems you encounter.

In [25]:
# Export to Excel — returns the path of the written file
filepath = bi.export.write_lci_excel(
    'Swiss garden foreground',
    dirpath=str(Path.cwd()),
)
print(f'Written to {filepath}')

Written to /Users/cmutel/Code/brightway/eta-teaching/lci-Swiss-garden-foreground.xlsx


In [26]:
# # Step 2 — Import
# bike_imp = bi.ExcelImporter(bicycle_filepath)

# # Step 3 — Default strategies
# bike_imp.apply_strategies()

# # Step 4 — Check statistics
# bike_imp.statistics()

In [27]:
# # Step 5 — Inspect unlinked and fix
# for edge in bike_imp.unlinked:
#     print(edge)

In [28]:
# # Step 6 — Match and Step 7 — Write
# # (fill in the correct database name(s) for your bicycle)
# bike_imp.match_database("Swiss garden foreground", fields=['name', 'location'])
# bike_imp.statistics()

# if bike_imp.all_linked:
#     bike_imp.write_database()
#     print("Done!")
# else:
#     print("Still some unlinked — inspect and fix before writing")

---

## Section 8 — Randonneur: Pre-computed Migrations

Writing custom fix functions works well for small problems. For systematic issues — such as **all names changing between ecoinvent 3.9 and 3.10** — you need pre-computed migration tables. The [randonneur_data](https://github.com/brightway-lca/randonneur_data) library is a registry of such tables.

In [29]:
import randonneur_data as rd

registry = rd.Registry()
print(f"Available migrations: {len(list(registry))}")
list(registry)[:10]

Available migrations: 57


['simapro-ecoinvent-3.10-cutoff',
 'simapro-ecoinvent-3.8-cutoff',
 'simapro-ecoinvent-3.9.1-cutoff',
 'ecoinvent-3.7.1-cutoff-ecoinvent-3.8-cutoff',
 'ecoinvent-3.8-cutoff-ecoinvent-3.9-cutoff',
 'ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff',
 'ecoinvent-3.8-biosphere-ecoinvent-3.9-biosphere',
 'ecoinvent-3.9.1-biosphere-ecoinvent-3.10-biosphere',
 'generic-brightway-units-normalization',
 'generic-brightway-unit-conversions']

In [30]:
# Sample one migration to see its structure
registry.sample('ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff')

{'replace': [{'source': {'name': 'hydrogen production, gaseous, petroleum refinery operation',
    'location': 'BR',
    'reference product': 'hydrogen, gaseous',
    'unit': 'kg'},
   'target': {'name': 'hydrogen production, gaseous, petroleum refinery operation',
    'location': 'BR',
    'reference product': 'hydrogen, gaseous, low pressure',
    'unit': 'kg'}},
  {'source': {'name': 'market for acrylic dispersion, with water, in 58% solution',
    'location': 'RER',
    'reference product': 'acrylic dispersion, with water, in 58% solution state',
    'unit': 'kg'},
   'target': {'name': 'market for acrylic dispersion, with water, in 58% solution state',
    'location': 'RER',
    'reference product': 'acrylic dispersion, with water, in 58% solution state',
    'unit': 'kg'}}],
 'disaggregate': [{'source': {'name': 'market for natural gas, low pressure',
    'location': 'RoW',
    'reference product': 'natural gas, low pressure',
    'unit': 'm3'},
   'targets': [{'name': 'market fo

Each migration entry maps old attribute values → new attribute values. To apply one to an importer:

```python
imp.randonneur(
    label='ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff',
    fields=['name', 'location', 'reference product'],
)
```

Randonneur migrations are particularly important when you have foreground data built against an older ecoinvent version and need to use it with a newer one. We will return to this in a later notebook.

---

## Summary

In this notebook you learned:

| Concept | Key point |
|---|---|
| BOM paradigm | Edges identified by attributes, not pointers — matching required |
| ETL pipeline | Extract → Transform → Load, with explicit strategies at each step |
| Custom strategies | Any function `f(data) → data` can be applied with `imp.apply_strategy(f)` |
| `link_iterable_by_fields` | Links only when exactly one match is found; use `edge_kinds` to separate technosphere and biosphere |
| Unlinked edges | Options: create new biosphere, add to existing biosphere, or drop (destructive) |
| Randonneur | Registry of pre-computed migrations for systematic name/unit changes |

In notebook 4 we will open the hood on the **matrix construction** that turns this linked graph into numbers.